In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import re

In [2]:
df = pd.read_csv('fake_news.csv')
df.head()

,id,title,author,date,source,url,language,country,topic,category,...,media_types,named_entity_count,char_count,word_count,citation_count,fact_check_verdict,fact_check_source,publication_timestamp,tokens_count,usability_flags
0,1,Government plan to reshape politics sector rev...,Chen Li,2024-10-08,LocalHerald,https://localherald.example/news/0,en,BR,politics,politics,...,"image,video",4,611,96,0,unverified,FactCheck.org,2024-10-08T00:00:00,96,"{""balanced_classes"": true, ""has_metadata"": tru..."
1,2,Government plan to reshape politics sector rev...,Priya Singh,2018-09-29,TrueVoice,https://truevoice.example/news/1,en,BR,politics,politics,...,none,5,1365,170,2,unverified,FactCheck.org,2018-09-29T00:00:00,170,"{""balanced_classes"": true, ""has_metadata"": tru..."
2,3,Government plan to reshape education sector re...,Liam O'Connor,2022-11-16,TrueVoice,https://truevoice.example/news/2,fr,GB,education,education,...,none,3,1392,171,2,true,FactCheck.org,2022-11-16T00:00:00,171,"{""balanced_classes"": true, ""has_metadata"": tru..."
3,4,Government plan to reshape politics sector rev...,Aisha Khan,2019-11-08,EpochView,https://epochview.example/news/3,hi,BR,politics,politics,...,none,3,938,117,1,unverified,NaN,2019-11-08T00:00:00,117,"{""balanced_classes"": true, ""has_metadata"": tru..."
4,5,How to protect yourself from education risks,NaN,2018-09-06,EpochView,https://epochview.example/news/4,hi,AU,education,education,...,"image,video",7,1663,207,3,true,Snopes,2018-09-06T00:00:00,207,"{""balanced_classes"": true, ""has_metadata"": tru..."


In [3]:
df.shape

(20000, 29)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 29 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   id                        20000 non-null  int64  
 1   title                     20000 non-null  object 
 2   author                    18065 non-null  object 
 3   date                      20000 non-null  object 
 4   source                    20000 non-null  object 
 5   url                       20000 non-null  object 
 6   language                  20000 non-null  object 
 7   country                   20000 non-null  object 
 8   topic                     20000 non-null  object 
 9   category                  20000 non-null  object 
 10  text                      20000 non-null  object 
 11  label                     20000 non-null  object 
 12  bias                      20000 non-null  object 
 13  source_reliability_score  20000 non-null  float64
 14  readab

In [5]:
df = df.drop(columns=['named_entity_count','char_count','word_count','citation_count','fact_check_verdict','fact_check_source','publication_timestamp','tokens_count','usability_flags','source_reliability_score','readability_score','sentiment_score','num_shares','num_comments'], axis=1)

In [6]:
df.head()

,id,title,author,date,source,url,language,country,topic,category,text,label,bias,has_media,media_types
0,1,Government plan to reshape politics sector rev...,Chen Li,2024-10-08,LocalHerald,https://localherald.example/news/0,en,BR,politics,politics,Tesla warns a shocking link between politics a...,fake,center,yes,"image,video"
1,2,Government plan to reshape politics sector rev...,Priya Singh,2018-09-29,TrueVoice,https://truevoice.example/news/1,en,BR,politics,politics,A recent peer-reviewed study published by IMF ...,real,right,no,none
2,3,Government plan to reshape education sector re...,Liam O'Connor,2022-11-16,TrueVoice,https://truevoice.example/news/2,fr,GB,education,education,A recent peer-reviewed study published by FDA ...,real,left,no,none
3,4,Government plan to reshape politics sector rev...,Aisha Khan,2019-11-08,EpochView,https://epochview.example/news/3,hi,BR,politics,politics,A recent peer-reviewed study published by IMF ...,real,left,no,none
4,5,How to protect yourself from education risks,NaN,2018-09-06,EpochView,https://epochview.example/news/4,hi,AU,education,education,A recent peer-reviewed study published by FBI ...,real,right,yes,"image,video"


In [7]:
print(df['source'].value_counts())
print(df['country'].value_counts())

source
RapidBulletin    2036
LocalHerald      2025
EpochView        2014
InsightWeekly    2012
DailyPost        2004
GlobalTimes      1992
CitizenReport    1986
TrueVoice        1982
WorldNewsNow     1982
MetroMirror      1967
Name: count, dtype: int64
country
BR    2074
FR    2034
IN    2011
GB    2010
ZA    1999
NG    1990
US    1984
DE    1975
AU    1965
CA    1958
Name: count, dtype: int64


In [8]:
print(df['language'].value_counts())
print(df['label'].value_counts())
print(df['topic'].value_counts())

language
es    5103
en    5002
hi    5001
fr    4894
Name: count, dtype: int64
label
fake          9216
real          8849
misleading    1935
Name: count, dtype: int64
topic
sports           2075
environment      2074
health           2046
politics         1997
science          1995
entertainment    1993
education        1988
finance          1963
technology       1954
crime            1915
Name: count, dtype: int64


In [9]:
print(df['category'].value_counts())
print(df['media_types'].value_counts())


category
sports           2075
environment      2074
health           2046
politics         1997
science          1995
entertainment    1993
education        1988
finance          1963
technology       1954
crime            1915
Name: count, dtype: int64
media_types
none           12591
image,video     2525
video           2486
image           2398
Name: count, dtype: int64


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   id           20000 non-null  int64 
 1   title        20000 non-null  object
 2   author       18065 non-null  object
 3   date         20000 non-null  object
 4   source       20000 non-null  object
 5   url          20000 non-null  object
 6   language     20000 non-null  object
 7   country      20000 non-null  object
 8   topic        20000 non-null  object
 9   category     20000 non-null  object
 10  text         20000 non-null  object
 11  label        20000 non-null  object
 12  bias         20000 non-null  object
 13  has_media    20000 non-null  object
 14  media_types  20000 non-null  object
dtypes: int64(1), object(14)
memory usage: 2.3+ MB


In [11]:
df.isnull().sum()

id                0
title             0
author         1935
date              0
source            0
url               0
language          0
country           0
topic             0
category          0
text              0
label             0
bias              0
has_media         0
media_types       0
dtype: int64

In [12]:
df['author'] = df['author'].fillna('Unknown')

In [13]:
df['author'].value_counts()

author
Unknown           1935
Daniel Park       1850
Sophia Rossi      1843
Aisha Khan        1828
Miguel Alvarez    1827
Noah Brown        1809
Alex Johnson      1803
Liam O'Connor     1797
Priya Singh       1795
Fatima Noor       1765
Chen Li           1748
Name: count, dtype: int64

In [14]:
df['text'].sample(3)

1596     A recent peer-reviewed study published by Appl...
16665    A recent peer-reviewed study published by UN a...
18778    A recent peer-reviewed study published by Tesl...
Name: text, dtype: object

## Text Cleaning

In [15]:
import re
import nltk
import string
from nltk.corpus import stopwords


stop_words = set(stopwords.words('english'))
puncuation  = string.punctuation

def cleaned_text(text):
    text = text.lower()

    # Remove URLs Tag is exist
    text = re.sub(r'https?://\S+|www\.\S+','', text)

    # Remove Punctuation
    text = text.replace(puncuation, '')

    # Stop word
    words = text.split()
    words = [word for word in words if word not in stop_words]
    return " ".join(words)

In [16]:
df['text'] = df['text'].apply(cleaned_text)

In [17]:
df['text']

0        tesla warns shocking link politics secret agen...
1        recent peer-reviewed study published imf inves...
2        recent peer-reviewed study published fda denie...
3        recent peer-reviewed study published imf uncov...
4        recent peer-reviewed study published fbi annou...
                               ...                        
19995    recent peer-reviewed study published cdc uncov...
19996    recent peer-reviewed study published investiga...
19997    recent peer-reviewed study published fda uncov...
19998    recent peer-reviewed study published un reveal...
19999    recent peer-reviewed study published nasa warn...
Name: text, Length: 20000, dtype: object

## Handling Label Data

In [94]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['encoded_label'] = le.fit_transform(df['label'])

## Convert Text into Numbers

In [145]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1,2), min_df = 2) # min_df = 2 means if that word occures less than 2 times so we can avoid that word
X = tfidf.fit_transform(df['text'])
y = df['encoded_label']

In [146]:
for word in list(tfidf.vocabulary_.keys())[:100]:
    if ' ' in word:
        print(word)

tesla warns
warns shocking
shocking link
link politics
politics secret
secret agendas
agendas insiders
insiders say
say this
this shock
shock you
you cdc
cdc reveals
reveals surprising
surprising link
you suggests
suggests shocking
you moderna
moderna denies
denies rare
rare link
you confirms
confirms controversial
controversial link
you eu
eu claims
claims alarming
alarming link
recent peer
peer reviewed
reviewed study
study published
published imf
imf investigates
investigates statistical
statistical evidence
evidence politics
politics reporting
reporting measured
measured changes
changes confidence
confidence intervals
intervals recent
published tesla
tesla denies
denies statistical
published cdc


In [147]:
tfidf.vocabulary_

{'tesla': np.int64(14026),
 'warns': np.int64(14063),
 'shocking': np.int64(14002),
 'link': np.int64(10561),
 'politics': np.int64(13945),
 'secret': np.int64(13998),
 'agendas': np.int64(9213),
 'insiders': np.int64(10535),
 'say': np.int64(13993),
 'this': np.int64(14037),
 'shock': np.int64(14000),
 'you': np.int64(14075),
 'cdc': np.int64(9742),
 'reveals': np.int64(13981),
 'surprising': np.int64(14021),
 'suggests': np.int64(14011),
 'moderna': np.int64(10574),
 'denies': np.int64(10440),
 'rare': np.int64(13973),
 'confirms': np.int64(9767),
 'controversial': np.int64(9777),
 'eu': np.int64(10459),
 'claims': np.int64(9755),
 'alarming': np.int64(9215),
 'tesla warns': np.int64(14036),
 'warns shocking': np.int64(14069),
 'shocking link': np.int64(14003),
 'link politics': np.int64(10568),
 'politics secret': np.int64(13947),
 'secret agendas': np.int64(13999),
 'agendas insiders': np.int64(9214),
 'insiders say': np.int64(10536),
 'say this': np.int64(13994),
 'this shock': np

In [168]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [149]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

lr = LogisticRegression(class_weight='balanced', max_iter=5000)
lr.fit(X_train, y_train)

LogisticRegression(class_weight='balanced', max_iter=5000)

In [160]:
y_pred = lr.predict(X_test)
accuracy_score(y_test, y_pred)

0.7615

In [161]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1829
           1       0.17      0.37      0.23       390
           2       0.81      0.60      0.69      1781

    accuracy                           0.76      4000
   macro avg       0.66      0.66      0.64      4000
weighted avg       0.84      0.76      0.79      4000



In [162]:
confusion_matrix(y_test, y_pred)

array([[1829,    0,    0],
       [   0,  145,  245],
       [   0,  709, 1072]])

In [164]:
dt = DecisionTreeClassifier(class_weight='balanced')
dt.fit(X_train, y_train)

DecisionTreeClassifier(class_weight='balanced')

In [166]:
y_pred1 = dt.predict(X_test)
accuracy_score(y_test, y_pred1)

0.82125

In [167]:
confusion_matrix(y_test, y_pred1)

array([[1829,    0,    0],
       [   0,   72,  318],
       [   0,  397, 1384]])